# F_04 재계산 — TF-IDF(liblinear) Paired Bootstrap 95% CI

이 노트북은 **11월 13일 변경된 D의 liblinear 예측값**으로 RQ1의 영어−대상언어 F1 차이의 CI를 재계산합니다. F_01~F_03은 건드리지 않습니다.

**준비 파일 (노트북과 같은 폴더에 두거나 아래 `INPUT_DIR`을 변경)**:
- `tfidf_test.npz`
- `tfidf_test.json`
- `tfidf_scores.csv`
- `en_test_Y.npy`
- `en_test_celex.json`
- (선택) `F_04_previous_lbfgs.csv` — 이전 분석과 비교할 때만 필요

방법: 아래 셀을 위에서부터 실행하세요. Google Colab에서 Drive를 사용할 경우 먼저 Drive를 마운트하고 `INPUT_DIR`을 설정하세요.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Recompute F_04 TF-IDF EN-vs-target paired bootstrap CIs from D's liblinear predictions.

This program does *not* retrain any model; it reuses each language's validation-selected
C, ngram and global threshold from the latest D tfidf_test.json.

Required files in --input-dir:
  tfidf_test.npz             probabilities of EN and 7 targets, plus aligned CELEX ids
  tfidf_test.json            D's solver and model settings/recorded F1
  tfidf_scores.csv           D's independent score table
  en_test_Y.npy             A's aligned 5000x21 binary ground truth (or .npy.bin)
  en_test_celex.json         A's 5000 CELEX IDs in prediction order
Optional --old-csv: previous lbfgs F_04 CSV, for a comparison file.

Run:
  python F_04_RQ1_TFIDF_CI.py --input-dir . --out-dir ./F_04_outputs \
      --old-csv F_04_previous_lbfgs.csv
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

LANGS = ('en', 'de', 'fr', 'el', 'fi', 'pl', 'bg', 'mt')
TARGETS = LANGS[1:]
N_BOOTSTRAP = 10_000
BOOTSTRAP_SEED = 42
CONFIDENCE_LEVEL = 0.95
BATCH_SIZE = 200
SOURCE_IDS = {
    'prediction_npz': '1r631ReMybi_RE4kv3OkJpOcNdnMKV5g7',
    'prediction_metadata': '1odwn88SW3unmSnlYHAjzHbxrezLM24ho',
    'original_score_table': '1bFza44aehN6V_d6xvlsft1a--0nwk-aV',
    'ground_truth': '11sH103TeQYfNzL8TvXL0KCJ6MbFrD_lL',
    'celex_ids': '1Ty2k_z8ihzZigspA4Eayu9Wno5yJFxXU',
}
CSV_COLUMNS = [
    'target_lang', 'english_tfidf_micro_f1', 'target_tfidf_micro_f1',
    'english_minus_target_tfidf_gap_pp', 'ci95_lower_pp', 'ci95_upper_pp',
    'ci95_excludes_zero', 'english_ngram', 'target_ngram', 'english_C',
    'target_C', 'english_threshold', 'target_threshold',
]


def resolve_input(folder: Path, file_name: str) -> Path:
    candidates = [folder / file_name, folder / (file_name + '.bin')]
    for c in candidates:
        if c.is_file():
            return c
    raise FileNotFoundError(f'Cannot find {file_name} in {folder}. Tried: {candidates}')


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda: f.read(1024 * 1024), b''):
            h.update(b)
    return h.hexdigest()


def micro_f1_from_counts(true_pos: np.ndarray, n_pred: np.ndarray, n_true: np.ndarray) -> np.ndarray:
    denom = n_pred + n_true
    return np.divide(2.0 * true_pos, denom, out=np.ones_like(denom, dtype=float), where=(denom != 0))


def calc_doc_counts(ground_truth: np.ndarray, p: np.ndarray, threshold: float) -> tuple[np.ndarray, np.ndarray]:
    predicted = p >= threshold
    return (
        np.count_nonzero(predicted & ground_truth, axis=1).astype(np.int16),
        np.count_nonzero(predicted, axis=1).astype(np.int16),
    )


def paired_bootstrap_gap(tp_docs: np.ndarray, pred_docs: np.ndarray,
                         true_docs: np.ndarray, n_boot: int = N_BOOTSTRAP,
                         seed: int = BOOTSTRAP_SEED,
                         batch_size: int = BATCH_SIZE) -> np.ndarray:
    """Return a 7 x B array of EN F1 minus target F1, in percentage points.

    One set of uniformly sampled CELEX document indices is shared across all 8
    languages at each replicate. The same RNG stream is used for every batch.
    """
    n_lang, n_docs = tp_docs.shape
    assert n_lang == len(LANGS) and pred_docs.shape == (n_lang, n_docs)
    rng = np.random.default_rng(seed)
    gaps = np.empty((len(TARGETS), n_boot), dtype=np.float64)
    for start in range(0, n_boot, batch_size):
        stop = min(n_boot, start + batch_size)
        sample_ix = rng.integers(0, n_docs, size=(stop - start, n_docs), dtype=np.int32)
        sample_truth_count = true_docs[sample_ix].sum(axis=1, dtype=np.int64)
        sample_tp = np.take(tp_docs, sample_ix, axis=1).sum(axis=2, dtype=np.int64)
        sample_pred_count = np.take(pred_docs, sample_ix, axis=1).sum(axis=2, dtype=np.int64)
        sample_micro_f1 = micro_f1_from_counts(sample_tp, sample_pred_count, sample_truth_count[None, :])
        gaps[:, start:stop] = 100.0 * (sample_micro_f1[0, None, :] - sample_micro_f1[1:, :])
    return gaps


def read_old_f04(path: Path | None) -> dict[str, dict]:
    if path is None:
        return {}
    with path.open('r', encoding='utf-8-sig', newline='') as f:
        rows = list(csv.DictReader(f))
    old = {r['target_lang'].lower(): r for r in rows}
    assert set(old) == set(TARGETS), 'The optional old F_04 CSV must have 7 target languages.'
    return old


def run_analysis(input_dir: Path, out_dir: Path, old_csv: Path | None = None) -> dict:
    input_dir = input_dir.resolve()
    out_dir.mkdir(parents=True, exist_ok=True)
    files = {k: resolve_input(input_dir, v) for k, v in {
        'npz': 'tfidf_test.npz', 'metadata': 'tfidf_test.json',
        'scores': 'tfidf_scores.csv', 'Y': 'en_test_Y.npy',
        'celex': 'en_test_celex.json',
    }.items()}

    meta = json.loads(files['metadata'].read_text(encoding='utf-8-sig'))
    assert meta['condition'] == 'tfidf' and meta['split'] == 'test'
    assert meta['solver'] == 'liblinear', f"Expected liblinear, got {meta['solver']!r}"
    assert meta['n_docs'] == 5000
    assert meta['label_column_order'] == list(range(21))
    with files['scores'].open('r', encoding='utf-8-sig', newline='') as f:
        scores = {r['language'].lower(): r for r in csv.DictReader(f)}
    assert set(scores) == set(LANGS)

    y = np.load(files['Y'], allow_pickle=False)
    assert y.shape == (5000, 21) and np.all((y == 0) | (y == 1))
    y = y.astype(bool)
    celex = json.loads(files['celex'].read_text(encoding='utf-8-sig'))
    assert len(celex) == 5000 and len(set(celex)) == 5000
    # The original D experiment hashes this exact, non-ASCII-escaped JSON serialization.
    celex_hash = hashlib.sha256(json.dumps(celex, ensure_ascii=False).encode('utf-8')).hexdigest()
    assert celex_hash == meta['celex_sha256'], 'CELEX ordering/hash does not match D metadata'

    with np.load(files['npz'], allow_pickle=False) as npz:
        assert set(npz.files) == set(LANGS) | {'ids'}, f'Unexpected NPZ keys: {npz.files}'
        npz_ids = npz['ids'].astype(str).tolist()
        assert npz_ids == celex, 'A ground-truth CELEX does not match D NPZ ID ordering'
        all_probabilities = {lang: npz[lang].copy() for lang in LANGS}

    true_per_doc = np.count_nonzero(y, axis=1).astype(np.int16)
    tp_docs, pred_docs = [], []
    micro_f1_by_lang = {}
    for lang in LANGS:
        p = all_probabilities[lang]
        assert p.shape == y.shape and np.isfinite(p).all()
        assert np.all((0.0 <= p) & (p <= 1.0))
        info = meta['results'][lang]
        row = scores[lang]
        assert row['ngram'] == info['ngram']
        for key in ('C', 'threshold', 'val_micro_f1', 'micro_f1'):
            assert np.isclose(float(row[key]), float(info[key]), atol=1e-12, rtol=0), (lang, key)
        tp, predicted_count = calc_doc_counts(y, p, float(info['threshold']))
        score = float(micro_f1_from_counts(tp.sum(dtype=np.int64), predicted_count.sum(dtype=np.int64),
                                           true_per_doc.sum(dtype=np.int64)))
        assert np.isclose(score, float(info['micro_f1']), atol=1e-12, rtol=0), (
            lang, score, info['micro_f1'])
        micro_f1_by_lang[lang] = score
        tp_docs.append(tp)
        pred_docs.append(predicted_count)

    tp_docs_matrix = np.stack(tp_docs, axis=0)
    pred_docs_matrix = np.stack(pred_docs, axis=0)
    boot_gap = paired_bootstrap_gap(tp_docs_matrix, pred_docs_matrix, true_per_doc)
    lower = np.quantile(boot_gap, (1.0 - CONFIDENCE_LEVEL) / 2.0, axis=1, method='linear')
    upper = np.quantile(boot_gap, (1.0 + CONFIDENCE_LEVEL) / 2.0, axis=1, method='linear')
    old = read_old_f04(old_csv)

    en_score = micro_f1_by_lang['en']
    en_meta = meta['results']['en']
    rows, detailed_results, comparison = [], {}, []
    for i, lang in enumerate(TARGETS):
        target = meta['results'][lang]
        point = 100.0 * (en_score - micro_f1_by_lang[lang])
        excludes_zero = bool((lower[i] > 0.0) or (upper[i] < 0.0))
        row = {
            'target_lang': lang,
            'english_tfidf_micro_f1': en_score,
            'target_tfidf_micro_f1': micro_f1_by_lang[lang],
            'english_minus_target_tfidf_gap_pp': point,
            'ci95_lower_pp': float(lower[i]),
            'ci95_upper_pp': float(upper[i]),
            'ci95_excludes_zero': excludes_zero,
            'english_ngram': en_meta['ngram'],
            'target_ngram': target['ngram'],
            'english_C': en_meta['C'],
            'target_C': target['C'],
            'english_threshold': en_meta['threshold'],
            'target_threshold': target['threshold'],
        }
        rows.append(row)
        detailed_results[lang] = {
            'english_tfidf': {key: en_score if key == 'micro_f1' else en_meta[key]
                              for key in ('micro_f1', 'ngram', 'C', 'threshold', 'val_micro_f1')},
            'target_tfidf': {key: micro_f1_by_lang[lang] if key == 'micro_f1' else target[key]
                             for key in ('micro_f1', 'ngram', 'C', 'threshold', 'val_micro_f1')},
            'english_minus_target_tfidf_gap_pp': point,
            'ci95_pp': {'lower': float(lower[i]), 'upper': float(upper[i])},
            'ci95_excludes_zero': excludes_zero,
        }
        if old:
            prev = old[lang]
            comparison.append({
                'target_lang': lang,
                'old_lbfgs_gap_pp': float(prev['english_minus_target_tfidf_gap_pp']),
                'new_liblinear_gap_pp': point,
                'new_minus_old_gap_pp': point - float(prev['english_minus_target_tfidf_gap_pp']),
                'old_lbfgs_ci_lower_pp': float(prev['ci95_lower_pp']),
                'new_liblinear_ci_lower_pp': float(lower[i]),
                'old_lbfgs_ci_upper_pp': float(prev['ci95_upper_pp']),
                'new_liblinear_ci_upper_pp': float(upper[i]),
                'old_lbfgs_ci_excludes_zero': prev['ci95_excludes_zero'],
                'new_liblinear_ci_excludes_zero': excludes_zero,
            })

    csv_path = out_dir / 'F_04_RQ1_TFIDF_EN_vs_Target_CI.csv'
    with csv_path.open('w', encoding='utf-8-sig', newline='') as f:
        w = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        w.writeheader()
        w.writerows(rows)
    json_path = out_dir / 'F_04_RQ1_TFIDF_EN_vs_Target_CI.json'
    structured = {
        'analysis': 'RQ1 TF-IDF Direct performance: English minus each target language',
        'split': 'test', 'dataset': 'MultiEURLEX', 'label_level': 1,
        'metric': 'micro_f1', 'f1_scale': '0_to_1',
        'gap_and_ci_unit': 'percentage_points',
        'n_docs': 5000, 'n_labels': 21, 'solver': 'liblinear',
        'bootstrap': {
            'method': 'paired_document_bootstrap_percentile',
            'resampling_unit': 'CELEX document', 'n_resamples': N_BOOTSTRAP,
            'seed': BOOTSTRAP_SEED, 'ci_level': CONFIDENCE_LEVEL,
            'same_resampled_indices_across_8_languages': True,
            'rng': 'numpy.random.default_rng(42)',
            'quantile_method': 'linear',
        },
        'selection': 'Each language n-gram, C and threshold chosen by validation by D (liblinear)',
        'sources': {k: f'https://drive.google.com/file/d/{v}/view' for k, v in SOURCE_IDS.items()},
        'input_sha256': {name: sha256_file(path) for name, path in files.items()},
        'validation': {
            'probabilities_recomputed_match_D_metadata': True,
            'probabilities_recomputed_match_D_tfidf_scores': True,
            'prediction_ids_match_A_en_test_celex': True,
            'celex_sha256_matches_D_metadata': True,
            'celex_sha256': celex_hash,
            'ground_truth_shape': [5000, 21],
            'solver_is_liblinear': True,
            'n_languages_verified': len(LANGS),
            'n_target_contrasts': len(TARGETS),
            'all_point_estimates_inside_own_ci': bool(all(
                lower[i] <= rows[i]['english_minus_target_tfidf_gap_pp'] <= upper[i]
                for i in range(len(TARGETS)))),
        },
        'cautions': [
            'Paired bootstrap captures test-document sampling variation only.',
            'No multiple-comparison adjustment across 7 contrasts.',
            'English-versus-target differences compare separately validation-tuned TF-IDF models.',
            'A CI containing zero does not establish equality of performance.',
            'This replaces F_04 lbfgs inputs only. F_01 to F_03 are unchanged.',
        ],
        'results': detailed_results,
    }
    json_path.write_text(json.dumps(structured, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

    comparison_path = None
    if comparison:
        comparison_path = out_dir / 'F_04_lbfgs_vs_liblinear_changes.csv'
        with comparison_path.open('w', encoding='utf-8-sig', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=list(comparison[0].keys()))
            writer.writeheader()
            writer.writerows(comparison)

    audit_path = out_dir / 'F_04_validation_log.json'
    audit = {
        'solver': 'liblinear', 'matched_documents': 5000,
        'languages': list(LANGS), 'comparison_count': len(TARGETS),
        'metadata_scores_recomputed_all_8_languages': True,
        'score_csv_matches_metadata_all_8_languages': True,
        'celex_match': True, 'shared_bootstrap_sample_indices': True,
        'bootstrap_replicates': N_BOOTSTRAP, 'seed': BOOTSTRAP_SEED,
        'source_ids': SOURCE_IDS,
        'input_sha256': structured['input_sha256'],
        'output_files': [csv_path.name, json_path.name] +
                        ([comparison_path.name] if comparison_path else []),
    }
    audit_path.write_text(json.dumps(audit, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

    print('F_04 TF-IDF RQ1, solver=liblinear, paired bootstrap B=10,000 seed=42')
    print('lang | EN F1    | Target F1 | gap pp    | 95% CI pp              | excludes 0')
    for r in rows:
        print(f"{r['target_lang']:4} | {r['english_tfidf_micro_f1']:.6f} | "
              f"{r['target_tfidf_micro_f1']:.6f}  | {r['english_minus_target_tfidf_gap_pp']:+.6f} | "
              f"[{r['ci95_lower_pp']:+.6f}, {r['ci95_upper_pp']:+.6f}] | {r['ci95_excludes_zero']}")
    print('Verified 8/8 D F1 scores, 5000 CELEX IDs, 7/7 point gaps; outputs:', out_dir)
    return structured


def main() -> None:
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--input-dir', type=Path, default=Path('.'))
    p.add_argument('--out-dir', type=Path, default=Path('./F_04_outputs'))
    p.add_argument('--old-csv', type=Path, default=None)
    args = p.parse_args()
    run_analysis(args.input_dir, args.out_dir, args.old_csv)



## 실행 설정
`INPUT_DIR`을 5개 소스 파일이 있는 경로로 설정하세요. `OUTPUT_DIR`에 결과 CSV·JSON·검증 로그가 저장됩니다.

In [ ]:
from pathlib import Path

INPUT_DIR = Path(".")  # 예: Path("/content/drive/MyDrive/기계학습_산출물/F_04_inputs")
OUTPUT_DIR = Path("./F_04_outputs")
OLD_CSV = INPUT_DIR / "F_04_previous_lbfgs.csv"
OLD_CSV = OLD_CSV if OLD_CSV.is_file() else None

result = run_analysis(INPUT_DIR, OUTPUT_DIR, OLD_CSV)
